In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

import pickle

from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

from scipy.io import mmread
from scipy.sparse import csr_matrix

from imblearn.over_sampling import SMOTE

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import make_scorer, f1_score
from imblearn.pipeline import Pipeline  # use imblearn pipeline if using SMOTE

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna/'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))

In [ ]:
assays = ['multiome', 'snmct']

In [ ]:
mct = sc.read_h5ad(f'{projdir}/h5ad/raw/mct.passqc_nuclei.h5ad')
multiome = sc.read_h5ad(f'{projdir}/h5ad/raw/multiome.passqc_nuclei.h5ad')

In [ ]:
mct

In [ ]:
multiome

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
clusters = pd.read_csv(f'{indir}/csv/clusters/rna_leiden_v7.csv', sep='\t', header=0, index_col=0)
multiome.obs['clusters'] = clusters.reindex(multiome.obs.index)['clusters_v7']
multiome.obs['coarse'] = clusters.reindex(multiome.obs.index)['coarse_v7']

In [ ]:
multiome.obs['coarse'].value_counts()

In [ ]:
multiome.obs['clusters'].value_counts()

In [ ]:
pcs = []
for donor in donors:
    tmp_pcs = pd.read_csv(f'{projdir}/csv/integrate/per_donor/{donor}.passqc_nuclei.pcs.csv', sep='\t', header=0, index_col=0)
    pcs.append(tmp_pcs)
merged_pcs = pd.concat(pcs)
merged_pcs.shape

In [ ]:
merged = mct.concatenate(multiome)
merged.obs.index = ['-'.join(x.split('-')[:-1]) for x in merged.obs.index]

In [ ]:
merged

In [ ]:
merged.obsm['X_pca'] = merged_pcs.reindex(merged.obs.index).values
merged.obs.index.isin(merged_pcs.index).sum()

In [ ]:
final_multiome = merged[merged.obs['batch'] == '1']
final_snmct = merged[merged.obs['batch'] == '0']

In [ ]:
X = final_multiome.obsm['X_pca']
y = final_multiome.obs['clusters']
train_size = 0.8
test_size = 0.2

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, train_size=train_size)
model = OneVsRestClassifier(XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
model.fit(X_train, y_train)

xgboost_pred = model.predict(X_test)
print(classification_report(y_test, xgboost_pred))

In [ ]:
X = final_multiome.obsm['X_pca']
y = final_multiome.obs['coarse']
train_size = 0.8
test_size = 0.2

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, train_size=train_size)
model = OneVsRestClassifier(XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
model.fit(X_train, y_train)

xgboost_pred = model.predict(X_test)
print(classification_report(y_test, xgboost_pred))

In [ ]:
clust_tmp = dict(zip(final_multiome.obs['coarse'].unique(), list(range(10))))
tmp_clust = dict(zip(list(range(9)), final_multiome.obs['coarse'].unique()))

final_multiome.obs['tmp_coarse'] = [clust_tmp[x] for x in final_multiome.obs['coarse']]

In [ ]:
# X = final_multiome.obsm['X_pca']
# y = final_multiome.obs['tmp_coarse']
# n_classes = y.unique().shape[0] 

# train_size = 0.8
# test_size = 0.2

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, train_size=train_size)
# model = XGBClassifier(
#     objective='multi:softprob',
#     num_class=n_classes,
#     use_label_encoder=False,
#     eval_metric='mlogloss',
#     random_state=42
# )

# model.fit(X_train, y_train)
# xgboost_pred = model.predict(X_test)

# ###
# final_test = [tmp_clust[x] for x in y_test]
# final_pred = [tmp_clust[x] for x in xgboost_pred]
# print(classification_report(final_test, final_pred))

In [ ]:
### include time as well

In [ ]:
features = np.hstack([final_multiome.obsm['X_pca'], final_multiome.obs['time_num'].values.reshape(-1, 1)])
features.shape

In [ ]:
features = np.hstack([final_multiome.obsm['X_pca'], final_multiome.obs['time_num'].values.reshape(-1, 1)])
X = features
y = final_multiome.obs['coarse']
train_size = 0.8
test_size = 0.2

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, train_size=train_size)
model = OneVsRestClassifier(XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
model.fit(X_train, y_train)

xgboost_pred = model.predict(X_test)
print(classification_report(y_test, xgboost_pred))

In [ ]:
xgboost_proba = model.predict_proba(X_test)
xgboost_proba.shape

In [ ]:
prob_df = pd.DataFrame(xgboost_proba, index=y_test.index, columns=model.classes_)

In [ ]:
class_df = pd.DataFrame(index=y_test.index)
class_df['ground_truth'] = y_test
class_df['xgboost_pred'] = xgboost_pred
final_df = pd.concat([class_df, prob_df], axis=1)
final_df.shape

In [ ]:
final_df.head()

In [ ]:
final_df.to_csv(f'{projdir}/csv/label_transfer/multiome_xgboost_preds.csv', sep='\t')

In [ ]:
features = np.hstack([final_multiome.obsm['X_pca'], final_multiome.obs['time_num'].values.reshape(-1, 1)])
X = features
y = final_multiome.obs['clusters']
train_size = 0.8
test_size = 0.2

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, train_size=train_size)
model = OneVsRestClassifier(XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
model.fit(X_train, y_train)

xgboost_pred = model.predict(X_test)
print(classification_report(y_test, xgboost_pred))

### can we improve this?
 - apply SMOTE to address class imbalances?

In [ ]:
final_multiome.obs['coarse'].value_counts()

In [ ]:
smote = SMOTE(random_state=42)
features = np.hstack([final_multiome.obsm['X_pca'], final_multiome.obs['time_num'].values.reshape(-1, 1)])
X = features
y = final_multiome.obs['coarse']
train_size = 0.8
test_size = 0.2
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, train_size=train_size)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
model = OneVsRestClassifier(XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
model.fit(X_train_resampled, y_train_resampled)

xgboost_pred = model.predict(X_test)
print(classification_report(y_test, xgboost_pred))

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure1'
merged.to_csv(f'{outdir}/multiome_snmct_clust_time_prop.csv', sep='\t')

### quick grid search

In [ ]:
features = np.hstack([final_multiome.obsm['X_pca'], final_multiome.obs['time_num'].values.reshape(-1, 1)])
X = features
y = final_multiome.obs['coarse']

In [ ]:
# %%time

# base_clf = XGBClassifier(
#     use_label_encoder=False,
#     eval_metric='logloss',
#     random_state=42,
#     n_jobs=-1
# )
# ovr_clf = OneVsRestClassifier(base_clf)

# param_grid = {
#     "estimator__max_depth": [3, 5, 7],
#     "estimator__learning_rate": [0.01, 0.1],
#     "estimator__n_estimators": [100, 200],
#     "estimator__scale_pos_weight": [1, 3]  # useful for class imbalance
# }

# # Optional: include SMOTE in the pipeline
# # pipeline = Pipeline([
# #     ("smote", SMOTE(random_state=42)),
# #     ("clf", ovr_clf)
# # ])
# pipeline = Pipeline([
#     ("clf", ovr_clf)
# ])

# # Define cross-validation scheme
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# # Define scorer (you can also try 'roc_auc_ovr' if using probability outputs)
# scorer = make_scorer(f1_score, average='macro')

# # Grid Search
# grid_search = GridSearchCV(
#     estimator=pipeline,
#     param_grid={f"clf__{k}": v for k, v in param_grid.items()},  # prefix with clf__ due to pipeline
#     scoring=scorer,
#     cv=cv,
#     verbose=2,
#     n_jobs=-1
# )

# # Fit
# grid_search.fit(X, y)

# # Results
# print("Best parameters:", grid_search.best_params_)
# # print("Best CV F1 score:", grid_search.best_score_)

# # Best model for final training/prediction
# best_model = grid_search.best_estimator_

In [ ]:
# grid_search.best_params_

# {'clf__estimator__learning_rate': 0.01,
#  'clf__estimator__max_depth': 3,
#  'clf__estimator__n_estimators': 100,
#  'clf__estimator__scale_pos_weight': 1}

In [ ]:
# ### for some reason performing worse? let's just stick to default

# features = np.hstack([final_multiome.obsm['X_pca'], final_multiome.obs['time_num'].values.reshape(-1, 1)])
# X = features
# y = final_multiome.obs['coarse']
# train_size = 0.8
# test_size = 0.2

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, train_size=train_size)
# # base_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, 
# #                          n_estimators=100, scale_pos_weight=1, max_depth=3, learning_rate=0.01)

# model = OneVsRestClassifier(base_clf)
# model.fit(X_train, y_train)

# xgboost_pred = model.predict(X_test)
# print(classification_report(y_test, xgboost_pred))

### okay, let's just take non-smote coarse as the final

In [ ]:
features = np.hstack([final_multiome.obsm['X_pca'], final_multiome.obs['time_num'].values.reshape(-1, 1)])
X = features
y = final_multiome.obs['coarse']
model = OneVsRestClassifier(XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
model.fit(X, y)

In [ ]:
snmct_features = np.hstack([final_snmct.obsm['X_pca'], final_snmct.obs['time_num'].values.reshape(-1, 1)])
snmct_features.shape

In [ ]:
final_snmct.obs['pred_coarse'] = model.predict(snmct_features)
final_snmct.obs['pred_coarse'].value_counts()

In [ ]:
final_snmct.obs['coarse_donor'] = [f'{x}_{y}' for x,y in zip(final_snmct.obs['pred_coarse'],
                                                             final_snmct.obs['donor'])]

In [ ]:
final_snmct.obs['coarse_donor'].value_counts().sort_index()

In [ ]:
select = ['donor', 'time_num', 'pred_coarse', 'coarse_donor']
final_snmct.obs[select].to_csv(f'{projdir}/csv/label_transfer/snmct_xgboost_v7.csv', sep='\t')

In [ ]:
!date